# ML Study Tracker: Unsupervised Learning — Practical Use Cases (v2, Beginner-Annotated)

**Unsupervised learning** means there is **no target column `y`** — no labels, no "right answers." The algorithm's job is to discover structure hidden in the features alone. The three big families, all covered here:

| Family | Question it answers | Sections |
|---|---|---|
| **Clustering** | "Which rows naturally belong together?" | 1–4 (K-Means, Hierarchical, DBSCAN, GMM) |
| **Dimensionality reduction** | "Can we compress many features into few, keeping the essence?" | 5–6 (PCA, t-SNE) |
| **Anomaly detection** | "Which rows don't belong at all?" | 7 (Isolation Forest) |

### The central difficulty: how do you know it worked?
In supervised learning, the test set grades you. Here there's no answer key, so we lean on:
- **Internal validation** — do clusters look tight and well-separated *by the data's own geometry*? (inertia, **silhouette score**, Davies-Bouldin)
- **Model-selection criteria** — statistical scores like **BIC** that reward fit but penalize complexity.
- **External validation** — *only in study settings*: when true labels secretly exist (wine cultivars, digit identities), we can compare clusters to them with **Adjusted Rand Index (ARI)**. ARI = 1 means clusters match labels perfectly, ARI ≈ 0 means no better than random assignment. In real unsupervised work you don't have this luxury — we use it here purely to check our understanding.
- **Downstream usefulness** — did the segments/components actually help the business or the next model? The ultimate judge.

### How to use this notebook
1. Run the **Common Imports** cell first.
2. Every numbered section is **self-contained** — run any one independently after imports.
3. Read the concept markdown before the code, then the **"Reading the output"** notes after.
4. One habit applies to nearly every section: **standardize features first**. Almost everything here is distance-based, and unscaled features silently rig the geometry (the Supervised notebook's KNN section demonstrates the damage).

### Key vocabulary used throughout
- **Silhouette score** (−1 to +1): for each point, compares its distance to its own cluster vs. the nearest other cluster. Near +1 = snugly inside its cluster; near 0 = sitting on a boundary; negative = probably in the wrong cluster. We report the average.
- **Inertia**: sum of squared distances from points to their cluster center. Lower = tighter, but it ALWAYS decreases as you add clusters — hence the "elbow" heuristic rather than simple minimization.

## Table of Contents
- [Common Imports & Configuration](#Common-Imports-&-Configuration)
- [1. K-Means — Customer Segmentation](#1.-K-Means-—-Customer-Segmentation)
- [2. Hierarchical Clustering — Dendrogram Analysis](#2.-Hierarchical-Clustering-—-Dendrogram-Analysis)
- [3. DBSCAN — Density Clusters & Noise](#3.-DBSCAN-—-Density-Clusters-&-Noise)
- [4. Gaussian Mixture Models — Soft Clustering & BIC](#4.-Gaussian-Mixture-Models-—-Soft-Clustering-&-BIC)
- [5. PCA — Dimensionality Reduction & Variance](#5.-PCA-—-Dimensionality-Reduction-&-Variance)
- [6. t-SNE — Non-Linear Visualization](#6.-t-SNE-—-Non-Linear-Visualization)
- [7. Isolation Forest — Anomaly Detection](#7.-Isolation-Forest-—-Anomaly-Detection)
- [Mini Project Tracker](#Mini-Project-Tracker)
- [Experiment Log](#Experiment-Log)

In [ ]:
# ============================================================
# Common Imports & Configuration
# Run this cell FIRST. Every section below assumes these exist.
# ============================================================
import numpy as np                     # numerical arrays and math
import pandas as pd                    # tabular data (DataFrames)
import matplotlib.pyplot as plt        # plotting

from sklearn.preprocessing import StandardScaler   # mean 0, std 1 per feature
from sklearn.pipeline import Pipeline

# Validation metrics for the label-free world:
#   silhouette_score      -> internal: cohesion vs separation, -1..+1
#   davies_bouldin_score  -> internal: avg cluster similarity, LOWER is better
#   adjusted_rand_score   -> external: agreement with true labels (study use only)
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

RANDOM_SEED = 42
# As in the Supervised notebook: no global np.random.seed(). Each section makes
# its own rng = np.random.default_rng(RANDOM_SEED) so every cell reproduces
# identically regardless of execution order.

## 1. K-Means — Customer Segmentation
**Dataset Target:** Synthetic RFM-Style Customer Behavior (Recency, Frequency, Monetary)

### The concept, in plain words
K-Means partitions data into **k** groups by repeating two steps until nothing moves (Lloyd's algorithm):
1. **Assign** each point to its nearest cluster center (centroid).
2. **Update** each centroid to the mean of its assigned points.

It's fast, simple, and the default first clustering attempt. Its assumptions — and therefore its failure modes — are: clusters are **round-ish, similar-sized blobs**, and **you must pick k in advance**.

### The business framing: RFM
Retail segmentation classically uses **R**ecency (days since last purchase — lower is better), **F**requency (purchases per period), **M**onetary (spend). We synthesize three personas — *Champions*, *Regulars*, *Dormant* — and check whether K-Means rediscovers them from the raw numbers alone.

### Picking k: two lenses, used together
- **Elbow (inertia) curve:** inertia always falls as k grows; look for the "elbow" where the drop suddenly flattens — extra clusters past that point buy little tightness.
- **Silhouette score:** unlike inertia it can *peak*, giving an actual argmax. When elbow and silhouette agree, be confident; when they disagree, profile both candidate k's and let interpretability decide.

⚠️ Two practical gotchas baked into the code: **standardize first** (Monetary is in hundreds, Recency in days — unscaled distance would be all about money), and use `n_init=10` (K-Means can converge to a poor local optimum from a bad random start; running 10 starts and keeping the best is cheap insurance).

### Study Checklist
- [ ] Centroid Allocation Optimization Loops (Lloyd's algorithm intuition)
- [ ] Euclidean Distance Scaling Dependency (standardize first, always)
- [ ] Elbow Curve via Inertia — and why it's ambiguous alone
- [ ] Silhouette Score Cohesion Evaluation
- [ ] Post-Clustering Business Profiling (translate clusters into personas)

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Practical Implementation: K-Means Segmentation, Elbow + Silhouette
# ============================================================
from sklearn.cluster import KMeans

rng = np.random.default_rng(RANDOM_SEED)

# --- Step 1: synthesize customers from three KNOWN personas ---
# Because we planted the personas, we can verify K-Means recovers them.
# Columns: Recency (days since last purchase), Frequency (orders/yr), Monetary ($/yr)
n_per = 300
champions = np.column_stack([rng.normal(5, 2, n_per),      # bought very recently
                             rng.normal(40, 8, n_per),     # buy often
                             rng.normal(900, 150, n_per)]) # spend a lot
regulars  = np.column_stack([rng.normal(20, 5, n_per),
                             rng.normal(12, 4, n_per),
                             rng.normal(300, 80, n_per)])
dormant   = np.column_stack([rng.normal(120, 30, n_per),   # haven't bought in months
                             rng.normal(2, 1, n_per),
                             rng.normal(60, 30, n_per)])
df = pd.DataFrame(np.vstack([champions, regulars, dormant]),
                  columns=['Recency_Days', 'Frequency', 'Monetary'])
# NOTE: no labels are kept — from here on, the algorithm is on its own.

# --- Step 2: standardize (Monetary ~ hundreds would dominate Euclidean distance) ---
X_scaled = StandardScaler().fit_transform(df)

# --- Step 3: choose k — run K-Means for k=2..7, record both criteria ---
ks, inertias, sils = range(2, 8), [], []
for k in ks:
    # n_init=10: run from 10 random starts, keep the best (avoids bad local optima)
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_SEED).fit(X_scaled)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_scaled, km.labels_))
    print(f"k={k} | inertia={km.inertia_:>8.1f} | silhouette={sils[-1]:.4f}")

# Plot both criteria side by side — the standard 'choosing k' picture
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(list(ks), inertias, marker='o'); ax[0].set(title='Elbow (inertia)', xlabel='k')
ax[1].plot(list(ks), sils, marker='o');     ax[1].set(title='Silhouette (peak = best)', xlabel='k')
plt.tight_layout(); plt.show()

# --- Step 4: fit final k=3 and PROFILE clusters in ORIGINAL units ---
# groupby-mean in raw units is what turns math clusters into business personas.
km = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_SEED).fit(X_scaled)
df['Cluster'] = km.labels_
print("\nCluster personas (means in original units):")
print(df.groupby('Cluster').mean().round(1))
print("\nCluster sizes:", df['Cluster'].value_counts().sort_index().tolist())

### Reading the output (Section 1)
- Both criteria point at **k=3**: the elbow flattens after 3, and silhouette peaks there (~0.70, comfortably high) — the algorithm recovered the three personas we planted.
- The profiling table is the deliverable: one cluster reads as low-recency/high-frequency/high-spend (*Champions*), one the opposite (*Dormant*), one in between. Cluster **numbers are arbitrary** (cluster 0 has no meaning); the *profile* is what you'd present.
- Real RFM data is messier — personas overlap and silhouette values of 0.2–0.4 are common and still useful. Also remember K-Means' blind spots: elongated, unequal-density, or nested clusters break it (Section 3 shows exactly that).

## 2. Hierarchical Clustering — Dendrogram Analysis
**Dataset Target:** Wine Dataset (Natural Multi-Group Chemistry Profiles)

### The concept, in plain words
**Agglomerative** (bottom-up) hierarchical clustering starts with every point as its own cluster, then repeatedly **merges the two closest clusters** until only one remains. The full merge history is drawn as a **dendrogram** — a tree where the height of each junction is the distance at which two clusters merged. Slice the tree horizontally at any height and the number of branches you cut = your number of clusters. Unlike K-Means, you don't commit to k up front; you choose it *after* seeing the structure.

### The choice that changes everything: linkage
"Distance between two clusters" needs a definition:
- **ward** — merge the pair that least increases within-cluster variance (tends to give compact, similar-size clusters; usually the best default).
- **complete** — distance between clusters = their *farthest* pair of points (compact but outlier-sensitive).
- **average** — mean of all cross-pair distances (a compromise; can produce straggly chains on some data).

The code compares all three on the same data — silhouette (internal) and ARI vs. the true cultivars (external, study-only) reveal how much linkage matters.

### Study Checklist
- [ ] Agglomerative Bottom-Up Merging Mechanics
- [ ] Linkage Criteria Comparison (ward vs. complete vs. average)
- [ ] Reading a Dendrogram — Cutting Height = Number of Clusters
- [ ] No Need to Pre-Specify k (vs. K-Means)
- [ ] External Validation Against Known Labels (ARI, learning-only)

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Practical Implementation: Agglomerative Clustering + Dendrogram
# ============================================================
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import load_wine
from scipy.cluster.hierarchy import dendrogram, linkage   # scipy draws dendrograms

# 178 wines x 13 chemical features. There ARE 3 true cultivars (grape types),
# but we hide them from the algorithm and use them only to grade it afterwards.
wine = load_wine()
X_scaled = StandardScaler().fit_transform(wine.data)   # distance-based -> scale first

# --- Step 1: same data, three linkage definitions ---
print("linkage  | silhouette (internal) | ARI vs true cultivars (external)")
for link in ['ward', 'complete', 'average']:
    labels = AgglomerativeClustering(n_clusters=3, linkage=link).fit_predict(X_scaled)
    print(f"{link:<8} | {silhouette_score(X_scaled, labels):.4f}                "
          f" | {adjusted_rand_score(wine.target, labels):.4f}")

# --- Step 2: the dendrogram — the algorithm's full merge history ---
# linkage() (scipy) computes the merge tree; dendrogram() draws it.
# truncate_mode='lastp' shows only the last 25 merges so the plot stays readable.
# HOW TO READ IT: y-axis = merge distance. Long vertical stems = well-separated
# groups. A horizontal cut across 3 stems = a 3-cluster solution.
plt.figure(figsize=(11, 4))
dendrogram(linkage(X_scaled, method='ward'), truncate_mode='lastp', p=25)
plt.title('Ward Linkage Dendrogram (last 25 merges)')
plt.xlabel('Sample index / (size of merged cluster)')
plt.ylabel('Merge distance')
plt.tight_layout(); plt.show()

### Reading the output (Section 2)
- **Linkage is not a detail**: ward reaches ARI ≈ 0.79 (strong agreement with the true cultivars) while average linkage collapses to ARI ≈ 0 — no better than random — *on identical data*. Algorithm choices inside the "same" method can swing results from excellent to useless.
- Note the silhouette values are modest (~0.28) even for ward, yet ARI is high. Silhouette measures geometric tightness in 13-D; clusters can be correct without being geometrically snug. No single validation number tells the whole story.
- In the dendrogram, the two or three **tallest vertical stems** near the top mark the natural major divisions — cutting just below them yields the 3-group solution. That visual, exploratory choice of k is hierarchical clustering's main gift over K-Means. (Its cost: O(n²) memory — impractical beyond ~tens of thousands of rows.)

## 3. DBSCAN — Density Clusters & Noise
**Dataset Target:** Non-Convex Synthetic Shapes (Where K-Means Fails)

### The concept, in plain words
DBSCAN defines clusters by **density**, not by distance to a center. Two parameters: `eps` (the radius of each point's neighborhood) and `min_samples` (how many neighbors make a neighborhood "dense"). Then:
- **Core point** — has ≥ `min_samples` neighbors within `eps`.
- **Border point** — within `eps` of a core point but not dense itself.
- **Noise** — neither. Labeled **−1** and belonging to *no* cluster.

Clusters grow by chaining core points whose neighborhoods overlap — so a cluster can be *any shape* a dense path can trace: crescents, rings, spirals.

### Why this section exists
K-Means can only produce **convex** regions (each point goes to its nearest centroid, carving space into flat-walled cells). On two interleaved crescent moons it fails structurally — no amount of tuning fixes it. DBSCAN handles them natively, plus gives you noise detection and no need to choose k. The price: `eps` is the new hard choice, and the sweep below shows how sharp its effect is.

### Study Checklist
- [ ] Density Reachability (core / border / noise points)
- [ ] eps & min_samples Sensitivity Sweep
- [ ] Noise Label (−1) Handling in Downstream Analysis
- [ ] Non-Convex Cluster Recovery (moons) — K-Means Failure Contrast
- [ ] No k Required, but eps Is the New Hard Choice

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Practical Implementation: DBSCAN on Non-Convex Data
# ============================================================
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_moons

# make_moons: two interleaved crescents — the canonical K-Means-breaker.
# y_true (which crescent each point belongs to) is kept ONLY for grading.
X_moons, y_true = make_moons(n_samples=600, noise=0.08, random_state=RANDOM_SEED)
X_scaled = StandardScaler().fit_transform(X_moons)

# --- Step 1: demonstrate the K-Means failure ---
# K-Means partitions space into convex cells around centroids; a crescent
# wrapping around another crescent cannot fit in a convex cell.
km_labels = KMeans(n_clusters=2, n_init=10, random_state=RANDOM_SEED).fit_predict(X_scaled)
print(f"K-Means ARI on moons: {adjusted_rand_score(y_true, km_labels):.4f}"
      "  <- structurally poor, no tuning can fix this")

# --- Step 2: eps sensitivity sweep ---
# Too small -> everything is 'noise' / fragments. Too large -> neighborhoods
# bridge the gap between crescents and merge everything into one blob.
print("\neps  | clusters found | noise points | ARI")
for eps in [0.1, 0.2, 0.3, 0.5]:
    db = DBSCAN(eps=eps, min_samples=5).fit(X_scaled)
    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)  # -1 isn't a cluster
    n_noise = (db.labels_ == -1).sum()
    print(f"{eps:.1f} | {n_clusters:>14} | {n_noise:>12} | {adjusted_rand_score(y_true, db.labels_):.4f}")

# --- Step 3: see it ---
best = DBSCAN(eps=0.2, min_samples=5).fit_predict(X_scaled)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(*X_scaled.T, c=km_labels, s=8, cmap='coolwarm')
ax[0].set_title('K-Means: slices straight across (wrong)')
ax[1].scatter(*X_scaled.T, c=best, s=8, cmap='coolwarm')
ax[1].set_title('DBSCAN eps=0.2: follows the crescents')
plt.tight_layout(); plt.show()

### Reading the output (Section 3)
- The sweep is the lesson: eps=0.1 shatters the data into ~27 fragments with heavy noise; **eps=0.2 nails it** (2 clusters, ARI ≈ 0.97); eps≥0.3 bridges the gap and merges everything into one cluster (ARI = 0, and note silhouette-style metrics can't even warn you here). DBSCAN's quality cliff around eps is steep — always sweep, never guess. (A principled starting point: the "k-distance elbow" heuristic — plot each point's distance to its 5th neighbor, sorted, and pick the knee.)
- The plots make K-Means' failure visceral: it draws a straight frontier through both crescents because that's all convex cells can do. Matching **algorithm assumptions to data shape** beats tuning every time.
- Practical note on the −1 label: noise points are *unassigned*, not "cluster −1". Downstream code that does `groupby(labels)` must handle them explicitly — a very common silent bug.

## 4. Gaussian Mixture Models — Soft Clustering & BIC
**Dataset Target:** Overlapping Synthetic Gaussian Populations

### The concept, in plain words
A GMM assumes the data was generated by **n overlapping Gaussian (bell-curve) blobs**, and estimates each blob's center, shape (covariance), and weight. Fitting uses the **EM algorithm**, an alternation directly analogous to K-Means' two steps:
- **E-step:** given current blobs, compute each point's *responsibility* — the probability it came from each blob.
- **M-step:** given responsibilities, re-estimate each blob's parameters as weighted averages.

Two upgrades over K-Means fall out of this:
1. **Soft assignments.** Every point gets a *probability per cluster* (`predict_proba`), not a hard label. A point that's 55/45 between two segments is flagged as genuinely ambiguous rather than forced into a box — often exactly what a business wants to know.
2. **Cluster shape.** With `covariance_type='full'`, blobs can be stretched ellipses of different sizes and orientations. (With `'spherical'` and equal weights, GMM essentially *becomes* K-Means — a nice way to see K-Means as a special case.)

### Choosing n_components without labels: BIC
The **Bayesian Information Criterion** = model fit (log-likelihood) *minus* a penalty for parameter count. More components always fit better, but BIC charges rent for them; **the minimum BIC** marks the sweet spot. This gives a *principled* answer to "how many clusters?" — arguably cleaner than eyeballing an elbow.

### Study Checklist
- [ ] EM Algorithm Intuition (E-step responsibilities, M-step parameter updates)
- [ ] Soft Assignments — predict_proba per cluster
- [ ] Covariance Types (full vs. diag vs. spherical; spherical ≈ K-Means)
- [ ] Model Selection via BIC/AIC (principled choice of n_components)
- [ ] When GMM Beats K-Means (elliptical, overlapping clusters)

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Practical Implementation: GMM with BIC Model Selection
# ============================================================
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs

# Four blobs with DIFFERENT spreads (cluster_std varies) — the kind of
# unequal, overlapping structure where GMM's flexible covariances shine.
# The true count (4) is hidden from the model; BIC must find it.
X_blob, _ = make_blobs(n_samples=900, centers=4,
                       cluster_std=[1.0, 2.5, 0.7, 1.8],
                       random_state=RANDOM_SEED)
X_scaled = StandardScaler().fit_transform(X_blob)

# --- Step 1: BIC sweep over candidate component counts ---
# BIC = -2*log-likelihood + penalty*(number of parameters). LOWER is better.
# AIC is similar with a lighter penalty (tends to pick more components).
print("n_components | BIC        | AIC")
bics = []
for n in range(1, 8):
    gmm = GaussianMixture(n_components=n,
                          covariance_type='full',   # each blob: own ellipse shape
                          random_state=RANDOM_SEED).fit(X_scaled)
    bics.append(gmm.bic(X_scaled))
    print(f"{n:>12} | {gmm.bic(X_scaled):>10.1f} | {gmm.aic(X_scaled):>10.1f}")

best_n = int(np.argmin(bics)) + 1     # +1 because the sweep started at n=1
print(f"\nBIC minimum at n_components = {best_n}  (true answer: 4)")

# --- Step 2: soft assignments — the defining difference vs K-Means ---
gmm = GaussianMixture(n_components=best_n, covariance_type='full',
                      random_state=RANDOM_SEED).fit(X_scaled)
proba = gmm.predict_proba(X_scaled)   # shape: (n_points, n_components)

# Points whose strongest membership is <70% sit in genuine overlap zones —
# information a hard-labeling algorithm simply throws away.
ambiguous = (proba.max(axis=1) < 0.7).sum()
print(f"Points with max membership < 70%: {ambiguous} of {len(X_scaled)}")
print("Example row of membership probabilities:", proba[0].round(3))

### Reading the output (Section 4)
- **BIC bottoms out exactly at 4** — the true blob count we hid — then rises as the penalty outweighs marginal fit. Watch AIC: with its lighter penalty it often stays flat or keeps drifting down; BIC's stronger penalty makes it the more conservative (and usually safer) chooser.
- The example membership row shows what "soft" means: probabilities across components summing to 1. Only a handful of points fall below 70% confidence here because the blobs are well-separated after scaling; on real customer data expect far more — and treat those ambiguous points as a *finding* (hybrid customers), not an annoyance.
- Caveat for honesty: GMM assumes Gaussian blobs. On the crescent moons of Section 3, it fails just like K-Means — soft labels don't fix wrong shape assumptions. Every model in this notebook has a geometry it believes in; your job is matching it to the data's.

## 5. PCA — Dimensionality Reduction & Variance
**Dataset Target:** Breast Cancer Wisconsin (30 Correlated Features)

### The concept, in plain words
**Principal Component Analysis** finds new axes for your data. PC1 is the direction along which the data varies *most*; PC2 the most-varying direction perpendicular to PC1; and so on. Each principal component is a weighted mix (a **linear combination**) of the original features. Because early components hoard the variance, you can often keep a handful of them and discard the rest — compressing 30 correlated features into ~10 nearly-lossless ones.

Two ways to think about it:
- *Statistics view:* eigendecomposition of the covariance matrix; eigenvalues = variance per component.
- *Geometry view:* rotate the cloud of points so its longest axis lines up with axis 1, second-longest with axis 2, ... then drop the short axes.

### Why scale FIRST (non-negotiable)
PCA chases variance. In raw units, a feature measured in thousands has enormous variance *purely from its units* and would hijack PC1. Standardizing makes "variance" mean *information*, not *unit size*.

### Choosing how many components
- **95% cumulative variance rule** — keep the smallest set of components explaining 95% of total variance (the scree curve below shows where that lands).
- **Loadings** — each component's weights over original features. Reading PC1's largest loadings tells you *what the dominant axis actually measures* — turning math back into meaning.

### Study Checklist
- [ ] Covariance Matrix Eigendecomposition Intuition
- [ ] Scaling BEFORE PCA (variance in raw units is meaningless across features)
- [ ] Explained Variance Ratio & Cumulative Scree Curve
- [ ] Choosing Components (95% variance rule vs. scree elbow)
- [ ] Loadings — Which Original Features Drive Each Component

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Practical Implementation: PCA — Variance Analysis & Projection
# ============================================================
from sklearn.decomposition import PCA
from sklearn.datasets import load_breast_cancer

# 569 tumors x 30 features. The 30 features are HIGHLY correlated
# (radius, perimeter, area all measure size) — ideal PCA territory:
# lots of redundancy to squeeze out.
cancer = load_breast_cancer()
X_scaled = StandardScaler().fit_transform(cancer.data)   # scale FIRST — always

# --- Step 1: fit full PCA and examine the variance ledger ---
pca = PCA().fit(X_scaled)                                # keep all 30 components
cumvar = np.cumsum(pca.explained_variance_ratio_)        # running total of variance
n95 = int(np.searchsorted(cumvar, 0.95)) + 1             # components to reach 95%
print(f"PC1 alone explains {pca.explained_variance_ratio_[0]:.1%} of all variance")
print(f"Components needed for 95%: {n95} of {X_scaled.shape[1]} -> "
      f"{X_scaled.shape[1]-n95} dimensions were mostly redundancy")

# --- Step 2: scree curve + 2D projection ---
# The 2D scatter is colored by the true diagnosis, which PCA NEVER SAW —
# if colors separate along PC1, the unsupervised axes align with the real signal.
X_2d = PCA(n_components=2).fit_transform(X_scaled)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(range(1, len(cumvar) + 1), cumvar, marker='o', ms=3)
ax[0].axhline(0.95, color='red', ls='--', lw=1)
ax[0].set(title='Cumulative Explained Variance (scree)', xlabel='# components',
          ylabel='cumulative ratio')
ax[1].scatter(X_2d[:, 0], X_2d[:, 1], c=cancer.target, s=6, cmap='coolwarm', alpha=0.6)
ax[1].set(title='2D projection (color = diagnosis, unseen by PCA)', xlabel='PC1', ylabel='PC2')
plt.tight_layout(); plt.show()

# --- Step 3: loadings — what does PC1 actually measure? ---
# components_[0] holds PC1's weight on each original feature.
loadings = pd.Series(pca.components_[0], index=cancer.feature_names)
print("\nLargest |loadings| on PC1 (the features defining the dominant axis):")
print(loadings.abs().sort_values(ascending=False).head(5).round(3))

### Reading the output (Section 5)
- **10 of 30 components carry 95% of the variance** — two-thirds of the measured dimensions were largely restating the other third. This is typical of hand-designed feature sets, where many features measure the same underlying thing.
- The 2D scatter shows malignant and benign tumors separating substantially **along PC1 alone** — remarkable, because PCA never saw the diagnosis. When unsupervised structure aligns with a label like this, the features contain strong signal (and a downstream classifier will have an easy time — exactly why PCA+SVM works so well in the Supervised notebook's Section 7).
- PC1's top loadings are all concavity/size measures — so PC1 ≈ *"overall tumor severity axis."* Naming your components via loadings is what makes PCA a communication tool, not just compression. One caution: PCA maximizes variance, not class separation — the alignment here is a happy property of this data, never a guarantee.

## 6. t-SNE — Non-Linear Visualization
**Dataset Target:** Handwritten Digits (64-Dim → 2-Dim Embedding)

### The concept, in plain words
PCA is linear — it can only rotate and project. **t-SNE** (t-distributed Stochastic Neighbor Embedding) is non-linear: it computes, in the original high-dimensional space, a probability that each pair of points are "neighbors," then arranges points on a 2-D canvas so those neighbor probabilities are matched as well as possible. Result: points that were close in 64-D end up close in 2-D — even if the structure is curved or tangled in ways PCA cannot flatten.

### The fine print (this is where people go wrong)
- **Only local structure is preserved.** Within-cluster layout is meaningful; **distances *between* clusters and cluster *sizes* are NOT** — a cluster appearing "far away" or "bigger" means nothing.
- **Perplexity** (~5–50) sets the effective neighborhood size the algorithm tries to preserve. Different perplexities give visibly different maps; always try a couple before trusting a story.
- **Visualization-only.** t-SNE has no `.transform()` for new points and its axes have no meaning — never feed t-SNE coordinates into a downstream model as features.
- **PCA pre-reduction first** (64→30 here) is standard practice: it strips noise dimensions and cuts computation substantially.

### Study Checklist
- [ ] Neighbor-Probability Matching Intuition (local structure preservation)
- [ ] Perplexity Sensitivity (effective neighborhood size)
- [ ] Why t-SNE Is Visualization-Only (inter-cluster distances/densities NOT meaningful)
- [ ] PCA Pre-Reduction for Speed on High-Dim Data
- [ ] Never Fit-Transform New Points (no out-of-sample transform)

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Practical Implementation: t-SNE Embedding of Digits
# ============================================================
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits

# 1,797 handwritten digits, each a 64-pixel vector. Ten TRUE groups (digits
# 0-9) exist, but t-SNE never sees them — we color by them afterwards to
# grade the map by eye.
digits = load_digits()
X_scaled = StandardScaler().fit_transform(digits.data)

# --- Step 1: PCA pre-reduction, 64 -> 30 dims ---
# Keeps the signal-bearing directions, drops noise dims, speeds up t-SNE.
X_pca = PCA(n_components=30, random_state=RANDOM_SEED).fit_transform(X_scaled)

# --- Step 2: t-SNE down to 2-D ---
X_tsne = TSNE(
    n_components=2,
    perplexity=30,           # effective neighborhood size — TRY 5 and 50 too
    init='pca',              # PCA initialization: more stable, reproducible layouts
    learning_rate='auto',    # modern recommended default
    random_state=RANDOM_SEED # t-SNE is stochastic; seed for reproducibility
).fit_transform(X_pca)

plt.figure(figsize=(7, 6))
sc = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=digits.target, s=6,
                 cmap='tab10', alpha=0.7)
plt.colorbar(sc, label='True digit (never shown to t-SNE)')
plt.title('t-SNE map of digits — read WITHIN clusters, not BETWEEN them')
plt.tight_layout(); plt.show()

### Reading the output (Section 6)
- Roughly ten well-separated islands emerge, and coloring confirms each island is (mostly) one digit — t-SNE recovered the class structure with **zero label information**. This is why it's the standard first look at any embedding space (word vectors, image features, single-cell genomics).
- Look for the instructive imperfections: a few points sit inside the *wrong* island — genuinely ambiguous handwriting (a 9 drawn like a 4). t-SNE surfaces label noise and hard cases beautifully.
- Discipline reminders while you admire the picture: island-to-island gaps and island sizes are artifacts of the layout, not facts about the data; and rerunning with perplexity 5 or 50 will redraw the map — structure that survives across perplexities is the part you can trust. (For a faster, transform-capable alternative used heavily in industry, look up **UMAP** — the ideas transfer directly.)

## 7. Isolation Forest — Anomaly Detection
**Dataset Target:** Synthetic Transaction Amounts with Injected Fraud

### The concept, in plain words
Most anomaly detectors model "normal" and flag what's far from it. Isolation Forest flips the logic with one elegant observation: **anomalies are easier to isolate.** Build a tree by splitting on random features at random thresholds; a point in the dense heart of the data needs many splits before it sits alone in a leaf, while an outlier gets cut off in just a few. Average the isolation depth over many random trees → shallow average depth = anomaly.

Why it's popular in practice: no distance metric (so no scaling worries), handles mixed feature scales, roughly linear time, and works purely from feature geometry — **no fraud labels needed to train**.

### The one parameter that matters: `contamination`
Your prior estimate of the anomaly *rate* (here 3%). It doesn't change the anomaly *scores* — it sets the score threshold so that ~3% of points get flagged. Set it from domain knowledge (historical fraud rate); the code's final plot shows the score distribution so you can see how sensitive the flags are to that cut.

### Evaluation honesty
Training is unsupervised, but *evaluating* recall/precision requires ground truth. Here we injected the fraud ourselves so we can grade honestly; in production you'd grade against investigator-confirmed cases, discovered after the fact.

### Study Checklist
- [ ] Isolation Principle (anomalies need fewer random splits to isolate)
- [ ] Contamination Parameter — Prior on Anomaly Rate
- [ ] Anomaly Score Distribution Reading
- [ ] Precision/Recall on Injected Anomalies (when ground truth exists)
- [ ] Unsupervised in Training, Evaluable Only with Labels

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Practical Implementation: Isolation Forest Fraud Detection
# ============================================================
from sklearn.ensemble import IsolationForest

rng = np.random.default_rng(RANDOM_SEED)

# --- Step 1: synthesize transactions — 970 normal + 30 injected frauds ---
# Features: [transaction amount, hour of day]
normal = np.column_stack([
    rng.normal(50, 15, 970),     # typical purchases ~ $50
    rng.normal(14, 4, 970),      # mostly daytime hours
])
fraud = np.column_stack([
    rng.normal(400, 100, 30),                             # suspiciously large amounts
    rng.choice([2, 3, 4], 30) + rng.normal(0, 0.5, 30),   # 2-4 AM activity
])
X = np.vstack([normal, fraud])
y_true = np.array([0] * 970 + [1] * 30)   # ground truth — kept ONLY for grading

# --- Step 2: fit UNSUPERVISED (the model never sees y_true) ---
iso = IsolationForest(
    n_estimators=200,        # number of random isolation trees
    contamination=0.03,      # prior: 'we expect ~3% of rows to be anomalous'
    random_state=RANDOM_SEED,
).fit(X)

# .predict() returns +1 = normal, -1 = anomaly. Convert to 0/1 for grading.
pred = (iso.predict(X) == -1).astype(int)

# --- Step 3: grade against the injected ground truth ---
tp = ((pred == 1) & (y_true == 1)).sum()
print(f"Flagged as anomalies: {pred.sum()}  |  True frauds caught: {tp}/30")
print(f"Precision: {tp / max(pred.sum(), 1):.4f}   (flags that were real fraud)")
print(f"Recall:    {tp / 30:.4f}   (frauds that got flagged)")

# --- Step 4: the score landscape ---
# decision_function: positive = normal side, negative = anomaly side.
# Separated humps = an easy problem; overlapping humps = expect FP/FN trade-offs.
scores = iso.decision_function(X)
plt.figure(figsize=(8, 3.5))
plt.hist(scores[y_true == 0], bins=50, alpha=0.6, label='normal')
plt.hist(scores[y_true == 1], bins=20, alpha=0.8, label='injected fraud')
plt.axvline(0, color='red', ls='--', lw=1, label='decision threshold')
plt.legend(); plt.title('Isolation Forest Anomaly Scores')
plt.xlabel('decision_function score'); plt.tight_layout(); plt.show()

### Reading the output (Section 7)
- Perfect precision *and* recall — because we made the frauds cartoonishly obvious (8× normal amounts, 3 AM). The score histogram shows why: two humps with clear water between them. **Real fraud lives in the overlap zone**; sophisticated fraudsters deliberately mimic the normal hump, which is why production systems layer supervised models (trained on confirmed cases) on top of unsupervised detectors like this.
- Sensitivity experiment worth running: set `contamination=0.10`. The model will dutifully flag ~100 points — the extra ~70 are normal transactions sacrificed to your inflated prior. Contamination is a *statement about the world*, and the model trusts you; wrong priors turn directly into false positives or missed fraud.
- Connecting the two notebooks: anomaly detection sits at the boundary of the paradigms — unsupervised training, but any *quantitative* evaluation quietly borrows labels. Knowing which side of that line each claim stands on is the mark of someone who actually understands the field.

## Mini Project Tracker

| Project | Topic | Dataset Benchmark | Model Baseline | Primary Metric | Status | Notes |
|---|---|---|---|---|---|---|
| **Customer Segmentation** | Clustering | Synthetic RFM | `KMeans(n_clusters=k)` | Inertia elbow + Silhouette | Not Started | Profile clusters in original units |
| **Wine Grouping** | Hierarchical | load_wine | `AgglomerativeClustering(ward)` | Silhouette / ARI | Not Started | Compare 3 linkage criteria |
| **Non-Convex Shapes** | Density Clustering | make_moons | `DBSCAN(eps, min_samples)` | ARI + noise count | Not Started | Contrast K-Means failure |
| **Overlapping Populations** | Probabilistic Clustering | make_blobs (varied std) | `GaussianMixture(full cov)` | BIC / soft membership | Not Started | BIC sweep for n_components |
| **Feature Compression** | Dim. Reduction | Breast Cancer | `PCA(n_components=0.95)` | Explained Variance | Not Started | Read PC1 loadings |
| **Digits Map** | Visualization | load_digits | `PCA(30) -> TSNE(2)` | Visual cluster separation | Not Started | Perplexity sensitivity check |
| **Fraud Flags** | Anomaly Detection | Synthetic Transactions | `IsolationForest(contamination)` | Precision/Recall on injected | Not Started | Score histogram reading |

## Experiment Log

| Date | Problem Context | Dataset Input | Model Configuration | Preprocessing Setup | Metric Tracked | Result | Next Progressive Steps |
|---|---|---|---|---|---|---|---|
| YYYY-MM-DD | Customer Segmentation | Synthetic RFM | KMeans (k=3, n_init=10) | StandardScaler | Silhouette | | Try k=4; compare with GMM soft labels |
| YYYY-MM-DD | Fraud Detection | Synthetic Transactions | IsolationForest (contam=0.03) | None (raw features) | Recall on injected | | Sweep contamination 0.01–0.05 |